# AQUA-SENS — Pipeline complet

> Notebook de documentation du systeme intelligent de surveillance de la qualite de l'eau.
>
> **Ecole Centrale Casablanca — Projet PLBD**

---

## Sommaire

1. [Chargement et traitement des donnees](#1)
2. [Tuning et entrainement des modeles](#2)
3. [Evaluation et comparaison](#3)
4. [Explicabilite SHAP](#4)
5. [Inference capteurs et diagnostic](#5)
6. [Logique de filtration](#6)
7. [Prevision LSTM](#7)
8. [Moteur d'alertes](#8)

In [ ]:
import sys
from pathlib import Path

# Ajouter src/ et racine au path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
print("Racine projet :", ROOT)

<a id="1"></a>
## 1. Chargement et traitement des donnees

Le dataset provient de Kaggle (Water Potability, 3276 echantillons). On selectionne 4 features contraintes par le cout des capteurs disponibles :
- **pH** : mesure via pH Meter V1.1
- **Solids (TDS)** : derive de la conductivite via TDS_EC_FACTOR (0.67)
- **Conductivity** : mesuree via Gravity TDS Meter V1.0
- **Turbidity** : mesuree via capteur TSW-20M

Le pipeline de traitement applique :
1. Ecretage IQR avec bornes physiques strictes (`TRAINING_BOUNDS`)
2. Imputation du pH par mediane conditionnelle au groupe Potability
3. Relabelisation : 1 = Non potable, 0 = Potable

In [ ]:
from config import DATA, FEATURES, TARGET, TRAINING_BOUNDS, PHYSICAL_BOUNDS, TDS_EC_FACTOR
from data_processing import raw_data_processing, preprocess_for_ml

print(f"Features diagnostic : {FEATURES}")
print(f"TDS_EC_FACTOR       : {TDS_EC_FACTOR}")
print(f"\nBornes d'entrainement (TRAINING_BOUNDS) :")
for k, v in TRAINING_BOUNDS.items():
    print(f"  {k:15s} : {v}")
print(f"\nBornes capteur Pi (PHYSICAL_BOUNDS) :")
for k, v in PHYSICAL_BOUNDS.items():
    print(f"  {k:15s} : {v}")

In [ ]:
# Charger et traiter les donnees
df = raw_data_processing(DATA["raw_path"])
print(f"\nShape apres traitement : {df.shape}")
print(f"\nDistribution de la cible :")
print(df[TARGET].value_counts().rename({0: "Potable", 1: "Non potable"}))

In [ ]:
# Visualisation des distributions par classe
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Distribution des features par classe de potabilite", fontsize=13, fontweight="bold")
colors = {"Potable": "#3BB273", "Non potable": "#E84855"}

for ax, feat in zip(axes, FEATURES):
    for label, grp in df.groupby(TARGET):
        name = "Potable" if label == 0 else "Non potable"
        ax.hist(grp[feat], bins=30, alpha=0.6, label=name, color=colors[name])
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

<a id="2"></a>
## 2. Evaluation et comparaison des modeles

8 modeles sont evalues via Grid Search (G-mean) puis validation croisee StratifiedKFold(10).
Les resultats sont stockes dans `outputs/reports/evaluation_report.csv`.

In [ ]:
# Charger le rapport d'evaluation
eval_path = ROOT / "outputs" / "reports" / "evaluation_report.csv"
eval_df = pd.read_csv(eval_path)

display_cols = ["model_key", "rank", "composite_score", "roc_auc_test", "gmean_test",
                "f1_test", "recall_test", "precision_test", "accuracy_test", "mcc_test", "threshold"]
display_cols = [c for c in display_cols if c in eval_df.columns]

print("Performances des modeles sur le jeu de test :")
eval_df[display_cols].style.background_gradient(subset=["composite_score"], cmap="YlOrRd")

In [ ]:
# Comparaison visuelle des metriques test
metrics = ["roc_auc_test", "f1_test", "recall_test", "precision_test", "accuracy_test"]
metrics = [m for m in metrics if m in eval_df.columns]

fig, axes = plt.subplots(1, len(metrics), figsize=(4*len(metrics), 5))
fig.suptitle("Comparaison des modeles — Metriques test", fontsize=13, fontweight="bold")
palette = ["#2E86AB", "#E84855", "#3BB273", "#F18F01", "#7B2D8B"]

for ax, metric in zip(axes, metrics):
    vals = eval_df.sort_values(metric, ascending=True)
    bars = ax.barh(vals["model_key"], vals[metric],
                   color=[palette[i % len(palette)] for i in range(len(vals))])
    ax.set_title(metric.replace("_test", "").replace("_", " ").upper(), fontsize=10)
    ax.set_xlim(0.4, 1.0)
    ax.axvline(0.5, color="gray", linestyle=":", linewidth=0.8)

plt.tight_layout()
plt.show()

<a id="4"></a>
## 3. Explicabilite SHAP

L'importance de chaque capteur dans la decision du modele est calculee via SHAP (TreeExplainer pour les modeles arborescents). Un resume JSON leger est genere pour le Raspberry Pi.

In [ ]:
import json

shap_path = ROOT / "outputs" / "reports" / "shap_summary.json"
if shap_path.exists():
    shap_data = json.loads(shap_path.read_text(encoding="utf-8"))
    print(f"Modele explique : {shap_data['model_key']} (rank #{shap_data['model_rank']})")
    print(f"Explainer       : {shap_data['explainer_type']}")
    print(f"Echantillons    : {shap_data['n_samples']}")
    print(f"\nClassement des features par importance SHAP :")

    feats = shap_data["feature_ranking"]
    vals  = shap_data["mean_abs_shap"]
    max_v = max(vals.values())

    fig, ax = plt.subplots(figsize=(8, 3))
    colors = ["#2E86AB", "#E84855", "#3BB273", "#F18F01"]
    bars = ax.barh(feats[::-1], [vals[f] for f in feats[::-1]],
                   color=[colors[i % len(colors)] for i in range(len(feats))])
    ax.set_xlabel("Importance SHAP moyenne |phi|")
    ax.set_title(f"Importances SHAP globales — {shap_data['model_key']}")
    plt.tight_layout()
    plt.show()
else:
    print("Resume SHAP non disponible. Lancez : python src/explainability.py")

<a id="5"></a>
## 4. Inference capteurs et diagnostic

En production sur le Raspberry Pi, le module `sensor_inference.py` :
1. Lit les 4 capteurs via ADS1115 (I2C) + DS18B20 (1-Wire)
2. Convertit les tensions en unites physiques (pH, TDS, conductivite, turbidite)
3. Applique le RobustScaler fitte a l'entrainement
4. Predit la potabilite via le ThresholdClassifier (seuil = 0.5)
5. Detecte les capteurs invalides (deconnectes ou satures)

In [ ]:
# Simulation d'un diagnostic (mode mock)
from sensor_inference import SensorPipeline, MockSensorReader

pipeline = SensorPipeline.from_saved_models(mock=True)
print(f"Modele charge : {pipeline.model}")
print(f"Seuil         : {pipeline.model.threshold}")

result = pipeline.run_once()
print(f"\n--- Resultat du diagnostic ---")
for k, v in result.items():
    if k != "raw_values":
        print(f"  {k:20s} : {v}")
print(f"\n  Valeurs capteurs :")
for f, v in result["raw_values"].items():
    print(f"    {f:15s} : {v}")

<a id="6"></a>
## 5. Logique de filtration

Le `FilterController` selectionne **un seul filtre** par cycle selon la priorite :

| Priorite | Condition | Filtre |
|---|---|---|
| P1 | Turbidite > 5 NTU | Sediments |
| P2 | pH hors [6.5, 8.5] ou contexte chimique | Charbon actif |
| P3 | Turbidite 2-5 NTU | Charbon compresse |
| P4 | Non potable sans regle P1-P3 | Charbon actif (defaut) + SHAP |

Recommandations supplementaires sans pompe : conductivite > 1500 uS/cm (warning), > 2700 uS/cm (critique).

In [ ]:
# Test des scenarios de filtration
from filter_controller import FilterController, FILTER_CONFIG

fc = FilterController(mock=True)

scenarios = [
    ("Turbidite haute",         {"ph":7.0, "Turbidity":8.0,  "Conductivity":500,  "Solids":335,  "Temperature":22}),
    ("pH bas",                  {"ph":5.5, "Turbidity":1.0,  "Conductivity":500,  "Solids":335,  "Temperature":22}),
    ("Turbidite moderee",       {"ph":7.0, "Turbidity":3.5,  "Conductivity":500,  "Solids":335,  "Temperature":22}),
    ("Conductivite critique",   {"ph":7.0, "Turbidity":1.0,  "Conductivity":3000, "Solids":2010, "Temperature":22}),
    ("Tout OK (potable)",       {"ph":7.2, "Turbidity":1.0,  "Conductivity":500,  "Solids":335,  "Temperature":22}),
    ("Non potable sans regle",  {"ph":7.0, "Turbidity":1.5,  "Conductivity":800,  "Solids":536,  "Temperature":22}),
]

print(f"{'Scenario':30s} | {'Filtre':30s} | {'Recos':5s} | Raison")
print("-" * 120)
for name, raw in scenarios:
    pot = 0 if name == "Tout OK (potable)" else 1
    dec = fc.decide({"raw_values": raw, "potability_now": pot})
    d = dec.to_dict()
    fname = d["filter_name"] or "Aucun"
    print(f"{name:30s} | {fname:30s} | {len(dec.recommendations):5d} | {dec.reason[:50]}")

<a id="7"></a>
## 6. Prevision LSTM

Le modele LSTM bi-couche (hidden=64, 2 layers, dropout=0.2) predit l'evolution des 5 parametres sur 24h a partir des 24 dernieres mesures horaires.

En production, le buffer horaire accumule les mesures et calcule la **mediane des 5 dernieres minutes** toutes les heures, pour lisser le bruit capteur tout en gardant la granularite d'entrainement.

<a id="8"></a>
## 7. Moteur d'alertes NM 03.7.001

Les alertes sont declenchees a partir des **valeurs prevues** (pas des valeurs actuelles) pour donner aux operateurs le temps d'intervenir.

3 niveaux : **CRITICAL** (seuil depasse), **WARNING** (approche du seuil), **INFO** (tendance)

Regles combinees :
- Turbidite elevee + chute pH → risque contamination microbiologique
- Conductivite monte sans turbidite → possible intrusion agricole
- pH en baisse progressive → acidification
- TDS/conductivite anormaux → contamination organique non ionique
- Temperature > 22C pendant > 6h → proliferation bacterienne

In [ ]:
# Simulation du moteur d'alertes sur des donnees prevues
sys.path.insert(0, str(ROOT / "prediction"))
from alerte_engine import AlertEngine

engine = AlertEngine()

# Simuler des previsions avec un pic de turbidite et une chute de pH
np.random.seed(42)
predicted = np.column_stack([
    np.linspace(7.2, 6.3, 24),          # pH en baisse
    np.full(24, 600) + np.random.normal(0, 20, 24),  # TDS stable
    np.full(24, 900) + np.random.normal(0, 30, 24),  # Conductivite stable
    np.concatenate([np.linspace(2, 8, 12), np.linspace(8, 3, 12)]),  # Pic turbidite
    np.full(24, 23) + np.random.normal(0, 0.5, 24),  # Temperature stable
])

features = ["ph", "Solids", "Conductivity", "Turbidity", "Temperature"]
alerts = engine.analyze(predicted, features, horizon_h=24)

print(engine.summary(alerts))